# 医疗对话数据集构建 MVP

## Goal

从 IMCS21、CHIP-MDCFNPC、MedDG 三种不同原始格式出发，每个来源固定抽样 50 个病例，完成统一结构、质量过滤、病例级划分，并生成 SFT/RL 候选数据。

## Setup

关键假设：

- 随机种子固定为 42；
- 每个来源抽样 50 条；
- 每个来源按 10 SFT / 30 RL / 10 holdout 划分；
- MedDG 本地缺少 train，因此使用 dev；
- CHIP 和 MedDG 的疾病实体属于弱标签，不伪装成病例级金标准。

In [1]:
import json
import sys
from collections import Counter
from pathlib import Path

current_dir = Path.cwd().resolve()
REPO_ROOT = current_dir if (current_dir / "self_scripts").exists() else current_dir.parent
sys.path.insert(0, str(REPO_ROOT))

from self_scripts.build_mvp_dataset import build_dataset

OUTPUT_DIR = REPO_ROOT / "data/processed_samples"
REPO_ROOT

PosixPath('/Users/elon2ge/workspace/MedAgent-RL')

## Steps

### 1. 执行确定性构建

In [2]:
manifest = build_dataset(
    repository_root=REPO_ROOT,
    output_dir=OUTPUT_DIR,
    sample_per_source=50,
    seed=42,
    meddg_split="dev",
)

print(json.dumps({
    "source_stats": manifest["source_stats"],
    "output_counts": manifest["output_counts"],
    "split_counts": manifest["split_counts"],
    "rl_readiness": manifest["rl_readiness"],
}, ensure_ascii=False, indent=2))

{
  "source_stats": {
    "IMCS21": {
      "available": 2472,
      "eligible_after_cleaning": 2472,
      "eligible_unique": 2472,
      "sampled": 50
    },
    "CHIP-MDCFNPC": {
      "available": 5000,
      "eligible_after_cleaning": 4723,
      "eligible_unique": 4628,
      "sampled": 50
    },
    "MedDG": {
      "available": 2000,
      "eligible_after_cleaning": 1978,
      "eligible_unique": 1978,
      "sampled": 50
    }
  },
  "output_counts": {
    "unified_cases": 150,
    "sft_turns": 334,
    "rl_seeds": 90,
    "holdout_cases": 30,
    "cold_start_input": 30
  },
  "split_counts": {
    "holdout": 30,
    "rl": 90,
    "sft": 30
  },
  "rl_readiness": {
    "CHIP-MDCFNPC:False": 30,
    "IMCS21:True": 30,
    "MedDG:False": 30
  }
}


### 2. 读取统一病例并检查结构

In [3]:
def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]

unified_cases = read_jsonl(OUTPUT_DIR / "mvp_unified_cases.jsonl")
sft_turns = read_jsonl(OUTPUT_DIR / "mvp_sft_seed_turns.jsonl")
rl_seeds = read_jsonl(OUTPUT_DIR / "mvp_rl_seed_cases.jsonl")
holdout_cases = read_jsonl(OUTPUT_DIR / "mvp_holdout_cases.jsonl")

print("统一病例字段：", list(unified_cases[0]))
print("对话 turn 字段：", list(unified_cases[0]["dialogue"][0]))
print("SFT 样本字段：", list(sft_turns[0]))
print("RL seed 字段：", list(rl_seeds[0]))

统一病例字段： ['case_id', 'source', 'source_split', 'source_record_id', 'language', 'self_report', 'dialogue', 'ground_truth', 'quality', 'content_fingerprint', 'mvp_split']
对话 turn 字段： ['turn_id', 'role', 'text', 'entities']
SFT 样本字段： ['example_id', 'case_id', 'source', 'prompt', 'response', 'response_origin', 'thinking_status']
RL seed 字段： ['case_id', 'source', 'prompt', 'ground_truth', 'patient_profile_seed', 'enhanced_description', 'enhanced_description_status', 'rl_ready']


### 3. 汇总来源、划分与标签质量

In [4]:
source_counts = Counter(case["source"] for case in unified_cases)
split_counts = Counter(case["mvp_split"] for case in unified_cases)
source_split_counts = Counter((case["source"], case["mvp_split"]) for case in unified_cases)
label_quality_counts = Counter(
    (case["source"], case["ground_truth"]["label_quality"])
    for case in unified_cases
)

print("来源：", dict(sorted(source_counts.items())))
print("划分：", dict(sorted(split_counts.items())))
print("来源 × 划分：", dict(sorted(source_split_counts.items())))
print("标签质量：", dict(sorted(label_quality_counts.items())))

来源： {'CHIP-MDCFNPC': 50, 'IMCS21': 50, 'MedDG': 50}
划分： {'holdout': 30, 'rl': 90, 'sft': 30}
来源 × 划分： {('CHIP-MDCFNPC', 'holdout'): 10, ('CHIP-MDCFNPC', 'rl'): 30, ('CHIP-MDCFNPC', 'sft'): 10, ('IMCS21', 'holdout'): 10, ('IMCS21', 'rl'): 30, ('IMCS21', 'sft'): 10, ('MedDG', 'holdout'): 10, ('MedDG', 'rl'): 30, ('MedDG', 'sft'): 10}
标签质量： {('CHIP-MDCFNPC', 'missing'): 37, ('CHIP-MDCFNPC', 'weak'): 13, ('IMCS21', 'gold'): 50, ('MedDG', 'missing'): 15, ('MedDG', 'weak'): 35}


## Checks

### 4. 检查去重、划分隔离和 RL readiness

In [5]:
case_ids = [case["case_id"] for case in unified_cases]
fingerprints = [case["content_fingerprint"] for case in unified_cases]
split_id_sets = {
    split: {case["case_id"] for case in unified_cases if case["mvp_split"] == split}
    for split in ("sft", "rl", "holdout")
}

assert len(unified_cases) == 150
assert len(set(case_ids)) == 150
assert len(set(fingerprints)) == 150
assert split_id_sets["sft"].isdisjoint(split_id_sets["rl"])
assert split_id_sets["sft"].isdisjoint(split_id_sets["holdout"])
assert split_id_sets["rl"].isdisjoint(split_id_sets["holdout"])

rl_ready = Counter((row["source"], row["rl_ready"]) for row in rl_seeds)
print("全部检查通过")
print("RL readiness：", dict(sorted(rl_ready.items())))

全部检查通过
RL readiness： {('CHIP-MDCFNPC', False): 30, ('IMCS21', True): 30, ('MedDG', False): 30}


## Next Steps

当前 MVP 已完成确定性数据工程阶段，下一步是执行 `mvp_cold_start_input.json`：

1. 每个病例调用一次 DeepSeek，为全部 Doctor turn 补写 `<think>`；
2. 模型只返回 `turn_id + thinking`，原始 Doctor answer 由本地程序保留；
3. `generate_mvp_cold_start_sft.py` 拼装与现有 train/val 对齐的最终 SFT JSON；
4. 人工抽检事实一致性后，再进行独立的 train/test 划分。

本次执行结果：150 个统一病例、334 条 SFT turn、90 个 RL seed、30 个 holdout；只有 30 个 IMCS21 RL seed 当前满足严格 RL-ready 条件。